In [ ]:
import zipfile

with zipfile.ZipFile('/content/Beef Freshness.v6-dataset-beef-v2.folder.zip', 'r') as z:
    z.extractall('/content/dataset')

print("Unzip Finish")

Unzip Finish


In [ ]:
import os

for split in ['train', 'valid', 'test']:
    path = f'/content/dataset/Beef Freshness.v6-dataset-beef-v2.folder/{split}'
    for cls in sorted(os.listdir(path)):
        count = len(os.listdir(f'{path}/{cls}'))
        print(f'{split}/{cls}: {count} ภาพ')

train/Fresh: 1208 ภาพ
train/Half-Fresh: 1100 ภาพ
train/Spoiled: 864 ภาพ
valid/Fresh: 164 ภาพ
valid/Half-Fresh: 163 ภาพ
valid/Spoiled: 126 ภาพ
test/Fresh: 85 ภาพ
test/Half-Fresh: 76 ภาพ
test/Spoiled: 66 ภาพ


In [ ]:
pip install torch torchvision scikit-learn

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import os

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {DEVICE}")

Using: cuda


In [ ]:
DATA_DIR = '/content/dataset/Beef Freshness.v6-dataset-beef-v2.folder'

train_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_transform)
val_dataset   = datasets.ImageFolder(f'{DATA_DIR}/valid', transform=val_transform)
test_dataset  = datasets.ImageFolder(f'{DATA_DIR}/test',  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f"Classes: {train_dataset.classes}")
print(f"Train: {len(train_dataset)} | Valid: {len(val_dataset)} | Test: {len(test_dataset)}")

Classes: ['Fresh', 'Half-Fresh', 'Spoiled']
Train: 3172 | Valid: 453 | Test: 227


#Class Weight

In [ ]:
labels = [label for _, label in train_dataset.samples]

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=labels
)

weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)
print(f"Class weights: {class_weights}")
# Fresh=0.xx, Half-Fresh=0.xx, Spoiled=0.xx

Class weights: [0.87527594 0.96121212 1.22376543]


#Model

In [ ]:
model = models.efficientnet_b0(weights="DEFAULT")

# Freeze layer
for param in model.parameters():
    param.requires_grad = False

model.classifier[1] = nn.Linear(1280, 3)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)

print("Model ready!")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 149MB/s]


Model ready!


#Train Model

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

print("Functions ready!")

Functions ready!


#Train แค่ Layer สุดท้าย (5 epochs)

In [ ]:
print("Phase 1: Training classifier only...")

for epoch in range(5):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc     = val_epoch(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/5 | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

print("Phase 1 done!")

Phase 1: Training classifier only...
Epoch 1/5 | Train Loss: 0.5991 Acc: 0.8181 | Val Loss: 0.3505 Acc: 0.9095
Epoch 2/5 | Train Loss: 0.3406 Acc: 0.9001 | Val Loss: 0.2654 Acc: 0.9249
Epoch 3/5 | Train Loss: 0.2766 Acc: 0.9098 | Val Loss: 0.2457 Acc: 0.9139
Epoch 4/5 | Train Loss: 0.2437 Acc: 0.9180 | Val Loss: 0.1789 Acc: 0.9558
Epoch 5/5 | Train Loss: 0.2105 Acc: 0.9319 | Val Loss: 0.1814 Acc: 0.9294
Phase 1 done!


#Phase 2: Unfreeze ทั้งหมด train ต่อ (10 epochs)

In [ ]:
# Unfreeze all
for param in model.parameters():
    param.requires_grad = True

# learning rate
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

best_val_acc = 0
patience = 3
wait = 0

print("Phase 2: Fine-tuning all layers...")

for epoch in range(10):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc     = val_epoch(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/10 | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    # Early Stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        wait = 0
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"Best model saved! Val Acc: {val_acc:.4f}")
    else:
        wait += 1
        print(f"No improvement {wait}/{patience}")
        if wait >= patience:
            print(f"Early stopping!")
            break

print(f"Phase 2 done! Best Val Acc: {best_val_acc:.4f}")

Phase 2: Fine-tuning all layers...
Epoch 1/10 | Train Loss: 0.0819 Acc: 0.9738 | Val Loss: 0.0260 Acc: 0.9912
Best model saved! Val Acc: 0.9912
Epoch 2/10 | Train Loss: 0.0256 Acc: 0.9921 | Val Loss: 0.0149 Acc: 0.9934
Best model saved! Val Acc: 0.9934
Epoch 3/10 | Train Loss: 0.0103 Acc: 0.9968 | Val Loss: 0.0164 Acc: 0.9934
No improvement 1/3
Epoch 4/10 | Train Loss: 0.0163 Acc: 0.9959 | Val Loss: 0.0160 Acc: 0.9912
No improvement 2/3
Epoch 5/10 | Train Loss: 0.0162 Acc: 0.9946 | Val Loss: 0.0208 Acc: 0.9934
No improvement 3/3
Early stopping!
Phase 2 done! Best Val Acc: 0.9934


#Test

In [ ]:
# load best_model.pt
model.load_state_dict(torch.load('best_model.pt'))
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

test_acc = correct / total
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

Test Accuracy: 0.9956 (99.56%)


#Save Weights

In [ ]:
torch.save(model.state_dict(), 'beef_model.pt')
print("Saved beef_model.pt")

Saved beef_model.pt


# Download

In [ ]:
from google.colab import files
files.download('beef_model.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>